# Structure Exposure Networks from the OpenView 3D graph

This notebook evaluates cumulative Top-2 SENs from the completed countywide OpenView 3D graph and the DINS-assessed population rebuilt in notebook 00. All analyses use an individual-edge floor of $\alpha=0.75$, with components formed strictly within each fire perimeter and destruction outcomes shuffled within fire. Outputs include the fire-specific map figure, a separate two-panel outcomes figure relating threshold to outcome diversity (a) and P50 SEN size to destruction (b), and a P25–P75 SEN-size sensitivity figure.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PACKAGE_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / 'environment.yml').exists() and (path / 'src').exists()
)
sys.path.insert(0, str(PACKAGE_ROOT))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import NullLocator, PercentFormatter

from src.analysis.fragility import FIRES, fit_5pl
from src.analysis.openview_3d_sen import (
    audit_openview_3d_run, build_fire_networks_from_openview_3d,
)
from src.analysis.sen import (
    class_shuffle_analysis, component_size_classes,
    component_size_count_summary, component_state,
    continuous_component_outcomes, coupling_at_probability,
    cumulative_top_n_component_state,
    cumulative_top_n_shared_fate_curve, shared_fate_curve,
)
from src.viz.sen import (
    INK, MID, WUI_INFLUENCE, WUI_INTERFACE, WUI_INTERMIX,
    _plot_network_map,
)
from src.viz.style import FIRE_COLORS, FIRE_MARKERS, apply_style
from src.figure_paths import save_figure as save_to_tier
from src.paths import DATA_DIR, DERIVED_3D, OPENVIEW_3D_RUN, OPENVIEW_FIRE_2D_RUN, RESULTS_DIR

apply_style()
RUN_TAG = 'openview3d_lariac6_2mi'
DERIVED = DERIVED_3D
ANALYSIS_PATH = DERIVED / 'analysis_openview3d.parquet'
RESULTS = RESULTS_DIR / RUN_TAG / 's2s_nc'
WUI_PATH = DATA_DIR / 'calfire_wui_la.gpkg'
PERIMETER_PATH = DATA_DIR / 'nx' / 'fire_perims.parquet'
RESULTS.mkdir(parents=True, exist_ok=True)
SHUFFLE_SEED = 20250803
FATE_SEED = 20250804
N_SIZE_SHUFFLES = int(os.environ.get('OPENVIEW_SEN_SIZE_SHUFFLES', '9999'))
N_FATE_SHUFFLES = int(os.environ.get('OPENVIEW_SEN_FATE_SHUFFLES', '499'))
SEN_N_NEIGHBORS = 2
SEN_EDGE_FLOOR_ALPHA = 0.75
SEN_TAG = 'ctop2_alpha075'

def save_figure(fig, stem, tier='exploratory', source=None):
    """Write one figure into its tier; only the threshold panels are main."""
    path = save_to_tier(fig, stem, tier, source=source, dpi=600)
    print('saved', path.with_suffix('.png'))
    return path


## Rebuild the SEN analysis on the new graph

For each OpenView building pair, the larger of the two directional whole-building view factors is the undirected pairwise coupling, $F_{ij}$. The threshold $F^*$ is the coupling associated with an absolute fitted destruction probability of 0.50 in the pooled exposed sample.

SENs are cumulative Top-2 components. Each building's two strongest incident couplings are selected, and the building qualifies when it has two distinct neighbors and $F_{i(1)} + F_{i(2)} \geq F^*$. Components use selected links whose endpoints both qualify and whose individual coupling satisfies $F_{ij} \geq \alpha F^*$ with $\alpha=0.75$, so a weak second link cannot join a component on the strength of its companion. As in the original analysis, only buildings inside the fire perimeter enter the graph; unassessed buildings remain as possible structural connectors but do not enter outcome statistics.

In [ ]:
analysis = pd.read_parquet(ANALYSIS_PATH)
required = {'graph_id', 'BLD_ID', 'fire', 'exposed', 'is_destroyed', 'F_destroyed_wmean'}
missing = required.difference(analysis.columns)
if missing:
    raise ValueError(f'OpenView analysis table is missing: {sorted(missing)}')

exposed = analysis.loc[analysis.exposed.eq(1)]
pooled_fit = fit_5pl(exposed.F_destroyed_wmean, exposed.is_destroyed)
F_IJ_THRESHOLD = coupling_at_probability(pooled_fit, 0.50)
threshold_table = pd.DataFrame([{
    'population': 'Pooled exposed OpenView 3D structures',
    'n': len(exposed),
    'p_min': pooled_fit['p_min'],
    'p_max': pooled_fit['p_max'],
    'k': pooled_fit['k'],
    'hill': pooled_fit['hill'],
    'asymmetry': pooled_fit['asymmetry'],
    'fragility_midpoint_F50': pooled_fit['f50'],
    'F_ij_threshold': F_IJ_THRESHOLD,
    'probability_equivalent': 0.50,
}])
threshold_table.to_csv(RESULTS / 'sen_threshold.csv', index=False)
display(audit_openview_3d_run(OPENVIEW_3D_RUN))
display(threshold_table.round(6))
print(f'Cumulative Top-2 node threshold: F_i(1) + F_i(2) >= {F_IJ_THRESHOLD:.6f}')
print(f'Individual edge floor: F_ij >= {SEN_EDGE_FLOOR_ALPHA:.2f} F* '
      f'= {SEN_EDGE_FLOOR_ALPHA * F_IJ_THRESHOLD:.6f}')


In [ ]:
networks = build_fire_networks_from_openview_3d(
    PACKAGE_ROOT, OPENVIEW_3D_RUN, FIRES, restrict_to_perimeter=True,
)

# Notebook 00 assigns graph-prefixed BLD_ID values where a source LARIAC
# identifier is missing or duplicated. Carry those exact analysis keys onto
# the network nodes so every assessed graph_id can receive its outcome.
analysis_ids = analysis.set_index('graph_id').BLD_ID
for fire in FIRES:
    nodes = networks[fire]['nodes']
    analysis_key = nodes.building_id.map(analysis_ids)
    nodes['BLD_ID'] = analysis_key.fillna(nodes.BLD_ID.astype(str))

states = {
    fire: cumulative_top_n_component_state(
        networks[fire], F_IJ_THRESHOLD, analysis,
        n_neighbors=SEN_N_NEIGHBORS,
        edge_floor_fraction=SEN_EDGE_FLOOR_ALPHA,
    )
    for fire in FIRES
}
network_summary = pd.DataFrame([states[fire]['summary'] for fire in FIRES])
network_summary.to_csv(
    RESULTS / f'sen_network_summary_{SEN_TAG}.csv', index=False,
)
for fire in FIRES:
    states[fire]['nodes'][[
        'BLD_ID', 'building_id', 'top_n_neighbor_count', 'top_n_sum',
        'qualifies_cumulative_n', 'component_id', 'component_size',
        'is_destroyed', 'assessed', 'geometry',
    ]].to_parquet(
        DERIVED / f'fig4{fire.lower()}_map_source_{SEN_TAG}.parquet',
        index=False,
    )
display(network_summary.round(4))


## Panels a/b — fire-specific SEN maps

Assessed structures are colored by cumulative Top-2 ($\alpha=0.75$) SEN size on one shared logarithmic scale. Gray structures are unassessed connectors. WUI boundaries and the fire perimeter are geographic context only.

In [ ]:
wui = gpd.read_file(WUI_PATH)
wui_interface = wui.loc[wui.WUI_DESC.eq('Interface')]
wui_intermix = wui.loc[wui.WUI_DESC.eq('Intermix')]
wui_influence = wui.loc[wui.WUI_DESC.eq('Influence Zone')]
perimeters = gpd.read_parquet(PERIMETER_PATH).set_index('FIRE_NAME')

largest_component = max(
    states[fire]['summary']['largest_component'] for fire in FIRES
)
size_scale_max = int(2 ** np.ceil(np.log2(largest_component)))
size_norm = LogNorm(vmin=1, vmax=size_scale_max)
size_cmap = 'viridis'

fig_ab, axes = plt.subplots(1, 2, figsize=(7.35, 3.75))
for ax, fire, panel in zip(axes, FIRES, ('a', 'b')):
    _plot_network_map(
        ax, states[fire], fire, wui_interface, wui_intermix,
        wui_influence, perimeters.loc[fire].geometry, size_norm,
        size_cmap, panel,
    )

legend_handles = [
    Patch(facecolor=plt.cm.viridis(.48), edgecolor='none',
          label='DINS-assessed building'),
    Patch(facecolor=plt.cm.Greys(.48), edgecolor='none',
          label='unassessed connector'),
    Line2D([0], [0], color=WUI_INFLUENCE, lw=1.1, label='Influence'),
    Line2D([0], [0], color=WUI_INTERMIX, lw=1.1, label='Intermix'),
    Line2D([0], [0], color=WUI_INTERFACE, lw=1.1, label='Interface'),
    Line2D([0], [0], color=INK, lw=1.2, label='fire perimeter'),
]
fig_ab.legend(
    handles=legend_handles, loc='lower center', ncol=6,
    bbox_to_anchor=(.5, .135), fontsize=5.7, handlelength=1.25,
    columnspacing=.9, frameon=False,
)
sm = plt.cm.ScalarMappable(norm=size_norm, cmap=size_cmap)
cbar_ax = fig_ab.add_axes([.16, .065, .68, .025])
cbar = fig_ab.colorbar(sm, cax=cbar_ax, orientation='horizontal')
color_ticks = 2 ** np.arange(0, int(np.log2(size_scale_max)) + 1, 2)
if color_ticks[-1] != size_scale_max:
    color_ticks = np.append(color_ticks, size_scale_max)
cbar.set_ticks(color_ticks)
cbar.set_ticklabels([f'{value:g}' for value in color_ticks])
cbar.set_label('Buildings per SEN', fontsize=6.6, labelpad=1.5)
cbar.ax.tick_params(labelsize=6, length=2, pad=1.5)
fig_ab.subplots_adjust(left=.02, right=.98, top=.88, bottom=.25, wspace=.12)
save_figure(fig_ab, f'fig4ab_sen_maps_openview3d_{SEN_TAG}')
fig_ab


## Outcome clustering within SENs

Across a filtration of probability-equivalent thresholds $F^*_p$, cumulative Top-2 components are rebuilt at each $p$ with the node rule $F_{i(1)} + F_{i(2)} \geq F^*_p$ and the edge floor $F_{ij} \geq 0.75F^*_p$. Shannon entropy measures within-SEN outcome diversity. Lower observed diversity than the within-fire shuffle envelope indicates stronger outcome clustering than expected from each fire's overall destruction rate.

In [ ]:
shared_fate = cumulative_top_n_shared_fate_curve(
    networks, analysis, pooled_fit, n_neighbors=SEN_N_NEIGHBORS,
    probabilities=np.arange(.05, .901, .025),
    n_shuffle=N_FATE_SHUFFLES, seed=FATE_SEED,
    edge_floor_fraction=SEN_EDGE_FLOOR_ALPHA,
)
shared_fate.to_csv(
    DERIVED / f'fig4c_shared_fate_source_{SEN_TAG}.csv', index=False,
)

fig_c, ax = plt.subplots(figsize=(4.0, 3.15))
for fire in FIRES:
    frame = shared_fate.loc[shared_fate.fire.eq(fire)]
    ax.plot(
        frame.p_destroyed_equivalent, frame.observed_diversity,
        color=FIRE_COLORS[fire], lw=1.35, marker=FIRE_MARKERS[fire],
        ms=2.7, markevery=4, label=fire.title(),
    )
pooled = shared_fate.loc[shared_fate.fire.eq('POOLED')]
ax.fill_between(
    pooled.p_destroyed_equivalent, pooled.null_lo, pooled.null_hi,
    color='#D5D5D2', alpha=.75, linewidth=0,
    label='within-fire shuffle, 95%',
)
ax.plot(pooled.p_destroyed_equivalent, pooled.null_mean,
        color=MID, lw=.9, ls='--')
ax.plot(pooled.p_destroyed_equivalent, pooled.observed_diversity,
        color=INK, lw=2.0, label='Pooled')
ax.axvline(.5, color='#999999', lw=.75, ls=':')
ax.text(
    .515, .965,
    'analysis threshold: $P(\\mathrm{destroyed})=0.50$\n'
    f'$F_{{i(1)}}+F_{{i(2)}} \\geq {F_IJ_THRESHOLD:.4f}$\n'
    f'$F_{{ij}} \\geq {SEN_EDGE_FLOOR_ALPHA:g}F^*$',
    transform=ax.get_xaxis_transform(), fontsize=6.15,
    color=INK, va='top', ha='left',
)
ax.set(
    xlabel='Cumulative Top-2 $P(\\mathrm{destroyed})$ equivalent',
    ylabel='Within-SEN outcome diversity', xlim=(.05, .90), ylim=(0, 1.02),
)
ax.grid(color='#E5E5E2', lw=.45)
ax.text(0, 1.07, 'c', transform=ax.transAxes, fontsize=9, fontweight='bold')
ax.text(.055, 1.07, 'Outcome clustering within SENs',
        transform=ax.transAxes, fontsize=8.2, fontweight='bold')
ax.legend(loc='lower right', fontsize=5.9, ncol=2,
          columnspacing=.8, handlelength=1.6)
fig_c.tight_layout()
plt.close(fig_c)
display(shared_fate[np.isclose(shared_fate.p_destroyed_equivalent, .5)].round(4))


## Joint outcomes figure — outcome diversity and destruction across SEN sizes

Assessed buildings are grouped by their absolute component size in doubling bins (1, 2, 3–4, 5–8, ...). Points are positioned at the geometric mean observed size, error bars are Wilson 95% intervals, and fire-specific ribbons show the within-fire shuffle distribution.

In [ ]:
node_classes, class_summary = component_size_classes(states)
null_summary, size_contrasts = class_shuffle_analysis(
    class_summary, n_shuffle=N_SIZE_SHUFFLES, seed=SHUFFLE_SEED,
)
continuous_outcomes = continuous_component_outcomes(states)
size_count_summary = component_size_count_summary(
    states, n_shuffle=N_SIZE_SHUFFLES, seed=SHUFFLE_SEED,
)
class_summary.to_csv(RESULTS / f'sen_component_size_destruction_{SEN_TAG}.csv', index=False)
null_summary.to_csv(RESULTS / f'sen_component_size_shuffle_{SEN_TAG}.csv', index=False)
size_contrasts.to_csv(RESULTS / f'sen_component_size_contrasts_{SEN_TAG}.csv', index=False)
continuous_outcomes.to_parquet(
    DERIVED / f'sen_continuous_component_outcomes_{SEN_TAG}.parquet', index=False,
)
size_count_summary.to_csv(DERIVED / f'fig4d_component_size_source_{SEN_TAG}.csv', index=False)

fig_d, ax = plt.subplots(figsize=(4.0, 3.15))
for fire in FIRES:
    frame = size_count_summary.loc[
        size_count_summary.fire.eq(fire)
    ].sort_values('size_bin_id')
    color = FIRE_COLORS[fire]
    ax.fill_between(frame.size_position, frame.null_lo, frame.null_hi,
                    color=color, alpha=.09, linewidth=0)
    ax.plot(frame.size_position, frame.null_mean,
            color=color, lw=.7, ls=':')
    yerr = np.vstack((
        frame.destroyed_share - frame.ci_lo,
        frame.ci_hi - frame.destroyed_share,
    ))
    ax.errorbar(
        frame.size_position, frame.destroyed_share, yerr=yerr,
        color=color, marker=FIRE_MARKERS[fire], ms=3.8, mfc='white',
        mew=.9, lw=1.3, elinewidth=.65, capsize=1.7, label=fire.title(),
    )
pooled_size = size_count_summary.loc[
    size_count_summary.fire.eq('POOLED')
].sort_values('size_bin_id')
pooled_yerr = np.vstack((
    pooled_size.destroyed_share - pooled_size.ci_lo,
    pooled_size.ci_hi - pooled_size.destroyed_share,
))
ax.errorbar(
    pooled_size.size_position, pooled_size.destroyed_share, yerr=pooled_yerr,
    color=INK, marker='o', ms=3.5, lw=1.9, elinewidth=.65,
    capsize=1.7, label='Pooled',
)
max_size = int(size_count_summary.bin_upper.max())
size_ticks = 2 ** np.arange(0, int(np.log2(max_size)) + 1)
ax.set_xscale('log', base=2)
ax.set_xticks(size_ticks, [f'{value:g}' for value in size_ticks])
ax.xaxis.set_minor_locator(NullLocator())
ax.set_xlim(.85, max_size * 1.12)
ax.set_ylim(.35, .98)
ribbon_edges = np.geomspace(1 / np.sqrt(2), size_scale_max * np.sqrt(2), 257)
ribbon_values = np.sqrt(ribbon_edges[:-1] * ribbon_edges[1:])
ax.pcolormesh(
    ribbon_edges, [.35, .361], ribbon_values[np.newaxis, :],
    cmap=size_cmap, norm=size_norm, shading='flat', rasterized=True, zorder=.2,
)
ax.set_xlabel('Buildings per SEN')
ax.set_ylabel('Buildings destroyed')
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis='y', color='#E5E5E2', lw=.45)
ax.text(0, 1.07, 'd', transform=ax.transAxes, fontsize=9, fontweight='bold')
ax.text(.055, 1.07, 'Destruction is elevated in larger SENs',
        transform=ax.transAxes, fontsize=8.2, fontweight='bold')
ax.legend(loc='lower right', fontsize=6.1, ncol=3,
          handlelength=1.4, columnspacing=.8)
fig_d.tight_layout()
plt.close(fig_d)
display(class_summary.round(4))
display(size_contrasts.round(4))

# Publication figure: the common fire encoding and a single legend make
# the size and threshold results directly comparable without annotations.
fig_cd, (ax_fate, ax_size) = plt.subplots(1, 2, figsize=(7.2, 3.15))

for fire in FIRES:
    frame = size_count_summary.loc[
        size_count_summary.fire.eq(fire)
    ].sort_values('size_bin_id')
    color = FIRE_COLORS[fire]
    ax_size.fill_between(
        frame.size_position, frame.null_lo, frame.null_hi,
        color=color, alpha=.09, linewidth=0,
    )
    ax_size.plot(
        frame.size_position, frame.null_mean, color=color, lw=.7, ls=':',
    )
    yerr = np.vstack((
        frame.destroyed_share - frame.ci_lo,
        frame.ci_hi - frame.destroyed_share,
    ))
    ax_size.errorbar(
        frame.size_position, frame.destroyed_share, yerr=yerr,
        color=color, marker=FIRE_MARKERS[fire], ms=3.6, mfc='white',
        mew=.9, lw=1.25, elinewidth=.6, capsize=1.5,
    )
ax_size.errorbar(
    pooled_size.size_position, pooled_size.destroyed_share, yerr=pooled_yerr,
    color=INK, marker='o', ms=3.4, lw=1.8, elinewidth=.6, capsize=1.5,
)
ax_size.set_xscale('log', base=2)
ax_size.set_xticks(size_ticks, [f'{value:g}' for value in size_ticks])
ax_size.xaxis.set_minor_locator(NullLocator())
ax_size.set_xlim(.85, max_size * 1.12)
ax_size.set_ylim(.35, .98)
ax_size.set(
    xlabel='SEN size at P50 threshold (buildings)',
    ylabel='Buildings destroyed (%)',
)
ax_size.yaxis.set_major_formatter(PercentFormatter(1))
ax_size.grid(axis='y', color='#E5E5E2', lw=.45)
ax_size.text(-.12, 1.035, 'b', transform=ax_size.transAxes,
             fontsize=9, fontweight='bold')

for fire in FIRES:
    frame = shared_fate.loc[shared_fate.fire.eq(fire)]
    ax_fate.plot(
        frame.p_destroyed_equivalent, frame.observed_diversity,
        color=FIRE_COLORS[fire], lw=1.25, marker=FIRE_MARKERS[fire],
        ms=2.7, markevery=4,
    )
ax_fate.fill_between(
    pooled.p_destroyed_equivalent, pooled.null_lo, pooled.null_hi,
    color='#D5D5D2', alpha=.75, linewidth=0,
)
ax_fate.plot(
    pooled.p_destroyed_equivalent, pooled.null_mean,
    color=MID, lw=.8, ls=':',
)
ax_fate.plot(
    pooled.p_destroyed_equivalent, pooled.observed_diversity,
    color=INK, lw=1.8,
)
ax_fate.axvline(.5, color='#777777', lw=.8, ls=':', zorder=.5)
ax_fate.set(
    xlabel='Connectivity threshold\n(destruction-probability equivalent)',
    ylabel='Within-SEN outcome diversity',
    xlim=(.05, .90), ylim=(0, 1.02),
)
ax_fate.grid(axis='y', color='#E5E5E2', lw=.45)
ax_fate.text(-.12, 1.035, 'a', transform=ax_fate.transAxes,
             fontsize=9, fontweight='bold')

joint_legend = [
    Line2D([], [], color=FIRE_COLORS[fire], marker=FIRE_MARKERS[fire],
           markerfacecolor='white', markeredgewidth=.9, lw=1.25,
           markersize=3.8, label=fire.title())
    for fire in FIRES
] + [
    Line2D([], [], color=INK, marker='o', lw=1.8, markersize=3.4,
           label='Pooled'),
    Patch(facecolor='#D5D5D2', edgecolor='none', alpha=.75,
          label='Within-fire shuffle (95%)'),
]
fig_cd.legend(
    handles=joint_legend, loc='upper center', bbox_to_anchor=(.5, .995),
    ncol=4, frameon=False, fontsize=6.2, handlelength=1.65,
    columnspacing=1.15,
)
fig_cd.subplots_adjust(left=.09, right=.985, top=.86, bottom=.22, wspace=.34)
save_figure(fig_cd, f'fig_sen_outcomes_openview3d_{SEN_TAG}')
fig_cd


## Threshold sensitivity — outcome diversity and SEN-size destruction

Panel a shows outcome diversity across the full connectivity-threshold filtration, with P50 marked by the dotted reference line. Panel b rebuilds cumulative Top-2 ($\alpha=0.75$) clusters at probability-equivalent P25, P50, and P75 thresholds and relates absolute SEN size to the assessed share destroyed. The three facets share axes so the contraction of cluster sizes under stricter thresholds remains visible.

In [ ]:
SIZE_SENSITIVITY_PROBABILITIES = (0.25, 0.50, 0.75)
size_sensitivity_frames = []
for probability in SIZE_SENSITIVITY_PROBABILITIES:
    threshold = coupling_at_probability(pooled_fit, probability)
    if np.isclose(probability, .50):
        threshold_summary = size_count_summary.copy()
    else:
        threshold_states = {
            fire: cumulative_top_n_component_state(
                networks[fire], threshold, analysis,
                n_neighbors=SEN_N_NEIGHBORS,
                edge_floor_fraction=SEN_EDGE_FLOOR_ALPHA,
            )
            for fire in FIRES
        }
        threshold_summary = component_size_count_summary(
            threshold_states, n_shuffle=N_SIZE_SHUFFLES,
            seed=SHUFFLE_SEED + int(probability * 100),
        )
    threshold_summary['probability_equivalent'] = probability
    threshold_summary['F_score_cutoff'] = threshold
    size_sensitivity_frames.append(threshold_summary)

size_threshold_sensitivity = pd.concat(
    size_sensitivity_frames, ignore_index=True,
)
size_threshold_sensitivity.to_csv(
    DERIVED / f'fig4a_component_size_threshold_sensitivity_{SEN_TAG}.csv',
    index=False,
)

sensitivity_max_size = int(size_threshold_sensitivity.bin_upper.max())
sensitivity_scale_max = int(
    2 ** np.ceil(np.log2(max(1, sensitivity_max_size)))
)
sensitivity_ticks = 2 ** np.arange(
    0, int(np.log2(sensitivity_scale_max)) + 1, 4,
)

fig_threshold_sensitivity = plt.figure(figsize=(7.2, 4.2))
outer_grid = fig_threshold_sensitivity.add_gridspec(
    1, 2, width_ratios=(1.55, 1.0), wspace=.28,
)
diversity_ax = fig_threshold_sensitivity.add_subplot(outer_grid[0, 0])
right_grid = outer_grid[0, 1].subgridspec(3, 1, hspace=.10)
sensitivity_axes = [
    fig_threshold_sensitivity.add_subplot(right_grid[0, 0]),
]
for index in range(1, 3):
    sensitivity_axes.append(fig_threshold_sensitivity.add_subplot(
        right_grid[index, 0], sharex=sensitivity_axes[0],
        sharey=sensitivity_axes[0],
    ))

for fire in FIRES:
    frame = shared_fate.loc[shared_fate.fire.eq(fire)]
    diversity_ax.plot(
        frame.p_destroyed_equivalent, frame.observed_diversity,
        color=FIRE_COLORS[fire], lw=1.25, marker=FIRE_MARKERS[fire],
        ms=2.7, markevery=4,
    )
diversity_ax.fill_between(
    pooled.p_destroyed_equivalent, pooled.null_lo, pooled.null_hi,
    color='#D5D5D2', alpha=.75, linewidth=0,
)
diversity_ax.plot(
    pooled.p_destroyed_equivalent, pooled.null_mean,
    color=MID, lw=.8, ls=':',
)
diversity_ax.plot(
    pooled.p_destroyed_equivalent, pooled.observed_diversity,
    color=INK, lw=1.8,
)
diversity_ax.axvline(.5, color='#777777', lw=.8, ls=':', zorder=.5)
diversity_ax.text(
    .512, .97, 'Analysis threshold',
    transform=diversity_ax.get_xaxis_transform(), ha='left', va='top',
    fontsize=5.8, color='#666666',
)
diversity_ax.text(
    .075, .885, 'Within-fire shuffle (95%)',
    ha='left', va='bottom', fontsize=5.8, color=MID,
)
diversity_ax.set(
    xlabel='Connectivity threshold (destruction-probability equivalent)',
    ylabel='Within-SEN outcome diversity',
    xlim=(.05, .90), ylim=(0, 1.02),
)
diversity_ax.grid(axis='y', color='#E5E5E2', lw=.45)
diversity_ax.text(
    -.055, 1.035, 'a', transform=diversity_ax.transAxes,
    fontsize=9, fontweight='bold',
)

for ax, probability in zip(
    sensitivity_axes, SIZE_SENSITIVITY_PROBABILITIES,
):
    threshold_frame = size_threshold_sensitivity.loc[
        np.isclose(
            size_threshold_sensitivity.probability_equivalent,
            probability,
        )
    ]
    for fire in FIRES:
        frame = threshold_frame.loc[
            threshold_frame.fire.eq(fire)
        ].sort_values('size_bin_id')
        yerr = np.vstack((
            frame.destroyed_share - frame.ci_lo,
            frame.ci_hi - frame.destroyed_share,
        ))
        ax.errorbar(
            frame.size_position, frame.destroyed_share, yerr=yerr,
            color=FIRE_COLORS[fire], marker=FIRE_MARKERS[fire],
            ms=3.0, mfc='white', mew=.8, lw=1.1,
            elinewidth=.5, capsize=1.3,
        )
    pooled_frame = threshold_frame.loc[
        threshold_frame.fire.eq('POOLED')
    ].sort_values('size_bin_id')
    pooled_error = np.vstack((
        pooled_frame.destroyed_share - pooled_frame.ci_lo,
        pooled_frame.ci_hi - pooled_frame.destroyed_share,
    ))
    ax.errorbar(
        pooled_frame.size_position, pooled_frame.destroyed_share,
        yerr=pooled_error, color=INK, marker='o', ms=2.8, lw=1.6,
        elinewidth=.5, capsize=1.3,
    )
    ax.set_xscale('log', base=2)
    ax.set_xticks(
        sensitivity_ticks, [f'{value:,}' for value in sensitivity_ticks],
    )
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlim(.82, sensitivity_scale_max * 1.18)
    ax.set_ylim(.35, .98)
    ax.text(
        .04, .88, f'P{int(probability * 100)}', transform=ax.transAxes,
        ha='left', va='top', fontsize=6.5, fontweight='bold',
    )
    ax.grid(axis='y', color='#E5E5E2', lw=.45)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(labelbottom=np.isclose(probability, .75))
sensitivity_axes[1].set_ylabel('Buildings destroyed (%)')
sensitivity_axes[2].set_xlabel('SEN size (buildings)')
sensitivity_axes[0].set_title(
    'Destruction increases with SEN size', loc='left',
    fontsize=7.2, fontweight='bold', pad=8,
)
sensitivity_axes[0].text(
    -.18, 1.18, 'b', transform=sensitivity_axes[0].transAxes,
    fontsize=9, fontweight='bold',
)
sensitivity_legend = [
    Line2D([], [], color=FIRE_COLORS[fire], marker=FIRE_MARKERS[fire],
           markerfacecolor='white', markeredgewidth=.8, lw=1.1,
           markersize=3.5, label=fire.title())
    for fire in FIRES
] + [
    Line2D([], [], color=INK, marker='o', lw=1.6, markersize=3.2,
           label='Pooled'),
]
fig_threshold_sensitivity.legend(
    handles=sensitivity_legend, loc='upper center',
    bbox_to_anchor=(.5, .995), ncol=3, frameon=False, fontsize=6.2,
    handlelength=1.6, columnspacing=1.2,
)
fig_threshold_sensitivity.subplots_adjust(
    left=.085, right=.985, top=.88, bottom=.14,
)
save_figure(
    fig_threshold_sensitivity,
    f'Fig3_sen_threshold_and_size_sensitivity_{SEN_TAG}',
    tier='main',
)
fig_threshold_sensitivity


## Supplementary — alternative link definitions

*Supplement — Supplementary Figs. S?–S?.*

Notebook 05 compares three ways of linking buildings into SENs. The figures above use cumulative Top-2 with $\alpha=0.75$. This section rebuilds the same three figures (maps, the outcome-diversity filtration with P50 SEN-size destruction, and the P25–P75 threshold sensitivity) for the other two definitions and for a hybrid of EEAT and cumulative Top-2, on the same within-perimeter fire networks, outcomes and within-fire shuffle nulls:

- **Every-Edge Above Threshold (EEAT)** keeps every edge with $F_{ij} \geq F^*$, with $F^*$ from the cumulative-dose fragility curve used above.
- **Top-2-pruned EEAT** keeps each building's two strongest incident edges and applies the threshold to each edge individually, without summing them. Its $F^*$ comes from a separate fragility curve fitted to each exposed building's maximum destroyed-neighbor pair dose (Fmax), as in notebook 05.
- **Anchored EEAT** keeps every EEAT edge and, in addition, edges with $0.75F^* \leq F_{ij} < F^*$ when an endpoint has at least one other edge with $F_{ij} \geq F^*$. A weaker link is admitted only where a strongly coupled building anchors it, as cumulative Top-2 lets a weak second link follow a strong first one, but without the two-link cap. `anchored_eeat_edge_mask(..., require_both=True)` gives the stricter variant in which both endpoints must be anchored.

Each method's P25, P50 and P75 thresholds are the corresponding probability equivalents of its own curve. The Top-2-pruned filtration runs on the graph restricted to every building's two strongest edges, which reproduces the method's P50 components exactly; anchored EEAT keeps every EEAT edge, so each EEAT component must lie inside one anchored component (both checked below). Each method keeps its own size scale, since component sizes differ between definitions. Outputs carry the method keys `eeat` and `eeat_top2_fmax` used in notebook 05, and `eeat_anchored`.

In [ ]:
from src.analysis.sen import (
    ANCHORED_EDGE_FLOOR_FRACTION, anchored_eeat_component_state,
    anchored_eeat_edge_mask, selected_edge_shared_fate_curve,
    top_n_eeat_component_state, top_n_pruned_network,
)

# Alternative link definitions, rebuilt on the same within-perimeter fire
# networks and probability-equivalent thresholds. EEAT keeps every edge with
# F_ij >= F* on the cumulative-dose curve. Top-2-pruned EEAT keeps each
# building's two strongest edges and applies the Fmax-curve threshold to each
# edge individually, without summing them. Anchored EEAT keeps every EEAT edge
# plus edges with F_ij >= 0.75 F* whose endpoint has another edge >= F*.
fmax_exposed = exposed.loc[
    exposed.dose_max_pair.notna() & exposed.dose_max_pair.gt(0)
]
if len(fmax_exposed) != len(exposed):
    raise ValueError(
        'Fmax and cumulative fragility fits must use the same exposed sample'
    )
fmax_fit = fit_5pl(fmax_exposed.dose_max_pair, fmax_exposed.is_destroyed)
pruned_networks = {
    fire: top_n_pruned_network(networks[fire], SEN_N_NEIGHBORS)
    for fire in FIRES
}


def _eeat_state(fire, threshold):
    return component_state(networks[fire], threshold, analysis)


def _pruned_eeat_state(fire, threshold):
    return top_n_eeat_component_state(
        networks[fire], threshold, analysis, n_neighbors=SEN_N_NEIGHBORS,
    )


def _anchored_state(fire, threshold):
    return anchored_eeat_component_state(networks[fire], threshold, analysis)


def _anchored_select(network, threshold):
    return anchored_eeat_edge_mask(network, threshold)


ALTERNATIVE_METHODS = {
    'eeat': {
        'label': 'Every-Edge Above Threshold (EEAT)',
        'rule': '$F_{ij} \\geq F^*$, cumulative-dose calibration',
        'fit': pooled_fit, 'fate_networks': networks, 'state': _eeat_state,
    },
    'eeat_top2_fmax': {
        'label': 'Top-2-pruned EEAT',
        'rule': ('two strongest edges per building, $F_{ij} \\geq F^*_{max}$, '
                 'Fmax calibration'),
        'fit': fmax_fit, 'fate_networks': pruned_networks,
        'state': _pruned_eeat_state,
    },
    'eeat_anchored': {
        'label': 'Anchored EEAT',
        'rule': ('$F_{ij} \\geq F^*$, plus $F_{ij} \\geq '
                 f'{ANCHORED_EDGE_FLOOR_FRACTION:g}F^*$ where an endpoint has '
                 'another edge $\\geq F^*$'),
        'fit': pooled_fit, 'select': _anchored_select,
        'state': _anchored_state,
    },
}

alternative_results = {}
for method_tag, method in ALTERNATIVE_METHODS.items():
    thresholds = {
        probability: coupling_at_probability(method['fit'], probability)
        for probability in SIZE_SENSITIVITY_PROBABILITIES
    }
    states_by_probability = {
        probability: {fire: method['state'](fire, threshold)
                      for fire in FIRES}
        for probability, threshold in thresholds.items()
    }
    method_states = states_by_probability[.50]
    for fire in FIRES:
        if 'select' in method:
            # Anchored EEAT keeps every EEAT edge, so each EEAT component must
            # lie inside a single anchored component.
            eeat_labels = component_state(
                networks[fire], thresholds[.50], analysis)['labels']
            nesting = pd.Series(method_states[fire]['labels']).groupby(
                eeat_labels).nunique()
            if nesting.gt(1).any():
                raise ValueError(f'{method_tag}: EEAT components are split')
            continue
        # The filtration runs on the pruned graph; at P50 it must reproduce the
        # components of the method's own state builder exactly.
        filtration_state = component_state(
            method['fate_networks'][fire], thresholds[.50], analysis,
        )
        if not np.array_equal(
                filtration_state['nodes'].component_size.to_numpy(),
                method_states[fire]['nodes'].component_size.to_numpy()):
            raise ValueError(f'{method_tag}: filtration graph differs at P50')

    method_summary = pd.DataFrame([
        {**method_states[fire]['summary'], 'method': method_tag}
        for fire in FIRES
    ])
    method_summary.to_csv(
        RESULTS / f'sen_network_summary_{method_tag}.csv', index=False,
    )
    for fire in FIRES:
        method_states[fire]['nodes'][[
            'BLD_ID', 'building_id', 'component_id', 'component_size',
            'is_destroyed', 'assessed', 'geometry',
        ]].to_parquet(
            DERIVED / f'fig4{fire.lower()}_map_source_{method_tag}.parquet',
            index=False,
        )

    if 'select' in method:
        method_fate = selected_edge_shared_fate_curve(
            networks, analysis, method['fit'], method['select'],
            probabilities=np.arange(.05, .901, .025),
            n_shuffle=N_FATE_SHUFFLES, seed=FATE_SEED,
        )
    else:
        method_fate = shared_fate_curve(
            method['fate_networks'], analysis, method['fit'],
            probabilities=np.arange(.05, .901, .025),
            n_shuffle=N_FATE_SHUFFLES, seed=FATE_SEED,
        )
    method_fate.to_csv(
        DERIVED / f'fig4c_shared_fate_source_{method_tag}.csv', index=False,
    )

    _, method_class_summary = component_size_classes(method_states)
    _, method_contrasts = class_shuffle_analysis(
        method_class_summary, n_shuffle=N_SIZE_SHUFFLES, seed=SHUFFLE_SEED,
    )
    method_class_summary.to_csv(
        RESULTS / f'sen_component_size_destruction_{method_tag}.csv',
        index=False,
    )
    method_contrasts.to_csv(
        RESULTS / f'sen_component_size_contrasts_{method_tag}.csv',
        index=False,
    )

    size_frames = []
    for probability, threshold in thresholds.items():
        frame = component_size_count_summary(
            states_by_probability[probability], n_shuffle=N_SIZE_SHUFFLES,
            seed=SHUFFLE_SEED + (0 if np.isclose(probability, .50)
                                 else int(probability * 100)),
        )
        size_frames.append(frame.assign(
            probability_equivalent=probability, F_score_cutoff=threshold,
        ))
    method_size_sensitivity = pd.concat(size_frames, ignore_index=True)
    method_size_sensitivity.to_csv(
        DERIVED
        / f'fig4a_component_size_threshold_sensitivity_{method_tag}.csv',
        index=False,
    )

    alternative_results[method_tag] = {
        'thresholds': thresholds, 'states': method_states,
        'summary': method_summary, 'shared_fate': method_fate,
        'contrasts': method_contrasts,
        'size_sensitivity': method_size_sensitivity,
    }

display(pd.concat(
    [result['summary'] for result in alternative_results.values()],
    ignore_index=True,
).round(4))
display(pd.concat(
    [result['contrasts'].assign(method=tag)
     for tag, result in alternative_results.items()],
    ignore_index=True,
).round(4))
display(pd.concat(
    [result['shared_fate'].loc[
        np.isclose(result['shared_fate'].p_destroyed_equivalent, .5)
    ].assign(method=tag) for tag, result in alternative_results.items()],
    ignore_index=True,
).round(4))

In [ ]:
# The three figures above, redrawn for each alternative link definition. Each
# method gets its own size scale, since EEAT components can be far larger, and
# a method line above the panels so the supplementary copies cannot be mistaken
# for the cumulative Top-2 originals.


def _method_heading(fig, method, threshold, y=.995):
    fig.text(.5, y, f"{method['label']} · {method['rule']} · "
             f"$F^* = {threshold:.4f}$ at P50",
             ha='center', va='top', fontsize=6.6, color='#444444')


def _size_scale(result):
    largest = max(result['states'][fire]['summary']['largest_component']
                  for fire in FIRES)
    scale_max = int(2 ** np.ceil(np.log2(max(2, largest))))
    return scale_max, LogNorm(vmin=1, vmax=scale_max)


def _destruction_ylim(frame):
    return min(.35, np.floor(20 * frame.ci_lo.min()) / 20), .98


def plot_alternative_maps(method_tag):
    method, result = ALTERNATIVE_METHODS[method_tag], alternative_results[method_tag]
    scale_max, norm = _size_scale(result)
    fig, axes = plt.subplots(1, 2, figsize=(7.35, 3.95))
    for ax, fire, panel in zip(axes, FIRES, ('a', 'b')):
        _plot_network_map(
            ax, result['states'][fire], fire, wui_interface, wui_intermix,
            wui_influence, perimeters.loc[fire].geometry, norm, size_cmap,
            panel,
        )
    fig.legend(
        handles=legend_handles, loc='lower center', ncol=6,
        bbox_to_anchor=(.5, .125), fontsize=5.7, handlelength=1.25,
        columnspacing=.9, frameon=False,
    )
    cbar_ax = fig.add_axes([.16, .06, .68, .024])
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=size_cmap),
                        cax=cbar_ax, orientation='horizontal')
    ticks = 2 ** np.arange(0, int(np.log2(scale_max)) + 1, 2)
    if ticks[-1] != scale_max:
        ticks = np.append(ticks, scale_max)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f'{value:,g}' for value in ticks])
    cbar.set_label('Buildings per SEN', fontsize=6.6, labelpad=1.5)
    cbar.ax.tick_params(labelsize=6, length=2, pad=1.5)
    _method_heading(fig, method, result['thresholds'][.50])
    fig.subplots_adjust(left=.02, right=.98, top=.84, bottom=.24, wspace=.12)
    save_figure(fig, f'SI_fig4ab_sen_maps_openview3d_{method_tag}',
                tier='supplementary')
    return fig


def _plot_size_destruction(ax, frame_all, *, ribbon=None, compact=False):
    for fire in FIRES:
        frame = frame_all.loc[frame_all.fire.eq(fire)].sort_values('size_bin_id')
        color = FIRE_COLORS[fire]
        if not compact:
            ax.fill_between(frame.size_position, frame.null_lo, frame.null_hi,
                            color=color, alpha=.09, linewidth=0)
            ax.plot(frame.size_position, frame.null_mean,
                    color=color, lw=.7, ls=':')
        yerr = np.vstack((frame.destroyed_share - frame.ci_lo,
                          frame.ci_hi - frame.destroyed_share))
        ax.errorbar(
            frame.size_position, frame.destroyed_share, yerr=yerr,
            color=color, marker=FIRE_MARKERS[fire],
            ms=3.0 if compact else 3.6, mfc='white',
            mew=.8 if compact else .9, lw=1.1 if compact else 1.25,
            elinewidth=.5 if compact else .6, capsize=1.3 if compact else 1.5,
        )
    pooled_frame = frame_all.loc[frame_all.fire.eq('POOLED')].sort_values(
        'size_bin_id')
    pooled_err = np.vstack((pooled_frame.destroyed_share - pooled_frame.ci_lo,
                            pooled_frame.ci_hi - pooled_frame.destroyed_share))
    ax.errorbar(
        pooled_frame.size_position, pooled_frame.destroyed_share,
        yerr=pooled_err, color=INK, marker='o', ms=2.8 if compact else 3.4,
        lw=1.6 if compact else 1.8, elinewidth=.5 if compact else .6,
        capsize=1.3 if compact else 1.5,
    )
    if ribbon is not None:
        scale_max, norm, low = ribbon
        edges = np.geomspace(1 / np.sqrt(2), scale_max * np.sqrt(2), 257)
        ax.pcolormesh(edges, [low, low + .011],
                      np.sqrt(edges[:-1] * edges[1:])[np.newaxis, :],
                      cmap=size_cmap, norm=norm, shading='flat',
                      rasterized=True, zorder=.2)
    ax.set_xscale('log', base=2)
    ax.xaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', color='#E5E5E2', lw=.45)


def _plot_diversity(ax, fate):
    for fire in FIRES:
        frame = fate.loc[fate.fire.eq(fire)]
        ax.plot(frame.p_destroyed_equivalent, frame.observed_diversity,
                color=FIRE_COLORS[fire], lw=1.25, marker=FIRE_MARKERS[fire],
                ms=2.7, markevery=4)
    pooled_fate = fate.loc[fate.fire.eq('POOLED')]
    ax.fill_between(pooled_fate.p_destroyed_equivalent, pooled_fate.null_lo,
                    pooled_fate.null_hi, color='#D5D5D2', alpha=.75,
                    linewidth=0)
    ax.plot(pooled_fate.p_destroyed_equivalent, pooled_fate.null_mean,
            color=MID, lw=.8, ls=':')
    ax.plot(pooled_fate.p_destroyed_equivalent, pooled_fate.observed_diversity,
            color=INK, lw=1.8)
    ax.axvline(.5, color='#777777', lw=.8, ls=':', zorder=.5)
    ax.set(xlim=(.05, .90), ylim=(0, 1.02),
           ylabel='Within-SEN outcome diversity')
    ax.grid(axis='y', color='#E5E5E2', lw=.45)


def plot_alternative_outcomes(method_tag):
    method, result = ALTERNATIVE_METHODS[method_tag], alternative_results[method_tag]
    scale_max, norm = _size_scale(result)
    p50 = result['size_sensitivity'].loc[
        np.isclose(result['size_sensitivity'].probability_equivalent, .50)]
    low, high = _destruction_ylim(p50)
    fig, (ax_fate, ax_size) = plt.subplots(1, 2, figsize=(7.2, 3.35))
    _plot_diversity(ax_fate, result['shared_fate'])
    ax_fate.set_xlabel(
        'Connectivity threshold\n(destruction-probability equivalent)')
    ax_fate.text(-.12, 1.035, 'a', transform=ax_fate.transAxes,
                 fontsize=9, fontweight='bold')
    _plot_size_destruction(ax_size, p50, ribbon=(scale_max, norm, low))
    ticks = 2 ** np.arange(0, int(np.log2(scale_max)) + 1,
                           1 if scale_max <= 256 else 2)
    ax_size.set_xticks(ticks, [f'{value:,g}' for value in ticks])
    ax_size.set_xlim(.85, scale_max * 1.12)
    ax_size.set_ylim(low, high)
    ax_size.set(xlabel='SEN size at P50 threshold (buildings)',
                ylabel='Buildings destroyed (%)')
    ax_size.text(-.12, 1.035, 'b', transform=ax_size.transAxes,
                 fontsize=9, fontweight='bold')
    fig.legend(handles=joint_legend, loc='upper center',
               bbox_to_anchor=(.5, .935), ncol=4, frameon=False,
               fontsize=6.2, handlelength=1.65, columnspacing=1.15)
    _method_heading(fig, method, result['thresholds'][.50])
    fig.subplots_adjust(left=.09, right=.985, top=.80, bottom=.20, wspace=.34)
    save_figure(fig, f'SI_fig_sen_outcomes_openview3d_{method_tag}',
                tier='supplementary')
    return fig


def plot_alternative_threshold_sensitivity(method_tag):
    method, result = ALTERNATIVE_METHODS[method_tag], alternative_results[method_tag]
    sensitivity = result['size_sensitivity']
    scale_max = int(2 ** np.ceil(np.log2(max(1, sensitivity.bin_upper.max()))))
    ticks = 2 ** np.arange(0, int(np.log2(scale_max)) + 1, 4)
    low, high = _destruction_ylim(sensitivity)
    fig = plt.figure(figsize=(7.2, 4.4))
    outer = fig.add_gridspec(1, 2, width_ratios=(1.55, 1.0), wspace=.28)
    diversity_ax = fig.add_subplot(outer[0, 0])
    right = outer[0, 1].subgridspec(3, 1, hspace=.10)
    size_axes = [fig.add_subplot(right[0, 0])]
    for index in range(1, 3):
        size_axes.append(fig.add_subplot(right[index, 0], sharex=size_axes[0],
                                         sharey=size_axes[0]))
    _plot_diversity(diversity_ax, result['shared_fate'])
    diversity_ax.set_xlabel(
        'Connectivity threshold (destruction-probability equivalent)')
    diversity_ax.text(.512, .97, 'Analysis threshold',
                      transform=diversity_ax.get_xaxis_transform(),
                      ha='left', va='top', fontsize=5.8, color='#666666')
    diversity_ax.text(-.055, 1.035, 'a', transform=diversity_ax.transAxes,
                      fontsize=9, fontweight='bold')
    for ax, probability in zip(size_axes, SIZE_SENSITIVITY_PROBABILITIES):
        _plot_size_destruction(
            ax, sensitivity.loc[np.isclose(sensitivity.probability_equivalent,
                                           probability)], compact=True)
        ax.set_xticks(ticks, [f'{value:,}' for value in ticks])
        ax.set_xlim(.82, scale_max * 1.18)
        ax.set_ylim(low, high)
        ax.text(.04, .88, f'P{int(probability * 100)}',
                transform=ax.transAxes, ha='left', va='top', fontsize=6.5,
                fontweight='bold')
        ax.tick_params(labelbottom=np.isclose(probability, .75))
    size_axes[1].set_ylabel('Buildings destroyed (%)')
    size_axes[2].set_xlabel('SEN size (buildings)')
    size_axes[0].set_title('Destruction across SEN sizes', loc='left',
                           fontsize=7.2, fontweight='bold', pad=8)
    size_axes[0].text(-.18, 1.18, 'b', transform=size_axes[0].transAxes,
                      fontsize=9, fontweight='bold')
    fig.legend(handles=sensitivity_legend, loc='upper center',
               bbox_to_anchor=(.5, .955), ncol=3, frameon=False, fontsize=6.2,
               handlelength=1.6, columnspacing=1.2)
    _method_heading(fig, method, result['thresholds'][.50])
    fig.subplots_adjust(left=.085, right=.985, top=.84, bottom=.13)
    save_figure(fig, f'SI_fig_sen_threshold_and_size_sensitivity_{method_tag}',
                tier='supplementary')
    return fig


alternative_figures = {}
for method_tag in ALTERNATIVE_METHODS:
    alternative_figures[method_tag] = [
        plot_alternative_maps(method_tag),
        plot_alternative_outcomes(method_tag),
        plot_alternative_threshold_sensitivity(method_tag),
    ]
    for fig in alternative_figures[method_tag]:
        display(fig)
        plt.close(fig)

In [ ]:
# Robustness copy: 2D OpenView cumulative Top-2 SENs, anchored at the
# average of the direct-2D and inverted-3D P50 midpoints.
FIRE_2D_RUN = OPENVIEW_FIRE_2D_RUN
F_2D_DIRECT_MIDPOINT = 0.2555667290936599
F_2D_INVERTED_3D_MIDPOINT = 0.16526373070
F_2D_MIDPOINT_AVERAGE = float(np.mean([
    F_2D_DIRECT_MIDPOINT, F_2D_INVERTED_3D_MIDPOINT,
]))


def _rescale_fit_to_p50(fit, target_p50):
    """Retain the fitted curve shape but set its P50 coupling scale."""
    scaled = dict(fit)
    current_p50 = coupling_at_probability(fit, .50)
    scaled['k'] = fit['k'] * float(target_p50) / current_p50
    scaled['f50'] = fit.get('f50', current_p50) * float(target_p50) / current_p50
    return scaled


def _threshold_size_result(method_fit, state_for, fate_for):
    thresholds = {
        probability: coupling_at_probability(method_fit, probability)
        for probability in SIZE_SENSITIVITY_PROBABILITIES
    }
    states_by_probability = {
        probability: {
            fire: state_for(fire, threshold) for fire in FIRES
        }
        for probability, threshold in thresholds.items()
    }
    size_frames = []
    for probability, threshold in thresholds.items():
        frame = component_size_count_summary(
            states_by_probability[probability],
            n_shuffle=N_SIZE_SHUFFLES,
            seed=SHUFFLE_SEED + int(probability * 100),
        )
        size_frames.append(frame.assign(
            probability_equivalent=probability,
            F_score_cutoff=threshold,
        ))
    return {
        'thresholds': thresholds,
        'states': states_by_probability[.50],
        'shared_fate': fate_for(method_fit),
        'size_sensitivity': pd.concat(size_frames, ignore_index=True),
    }


edges_2d_all = pd.read_parquet(
    FIRE_2D_RUN / 'building_edges.parquet',
    columns=['building_i', 'building_j', 'vf_i_to_j', 'vf_j_to_i'],
)
edges_2d_all['F_ij'] = np.maximum(
    edges_2d_all.vf_i_to_j, edges_2d_all.vf_j_to_i,
).clip(upper=1)
networks_2d = {}
for fire in FIRES:
    nodes = networks[fire]['nodes'].copy()
    node_index = nodes.set_index('building_id').node
    keep = (
        edges_2d_all.building_i.isin(node_index.index)
        & edges_2d_all.building_j.isin(node_index.index)
    )
    edges = edges_2d_all.loc[
        keep, ['building_i', 'building_j', 'F_ij']
    ].copy()
    edges['u'] = edges.building_i.map(node_index).astype(np.int64)
    edges['v'] = edges.building_j.map(node_index).astype(np.int64)
    edges = edges.rename(columns={
        'building_i': 'bld_a', 'building_j': 'bld_b',
    })
    edges['ssd_m'] = np.nan
    edges['has_contact'] = False
    networks_2d[fire] = {
        **networks[fire], 'nodes': nodes,
        'edges': edges[[
            'bld_a', 'bld_b', 'u', 'v', 'F_ij', 'ssd_m', 'has_contact',
        ]].reset_index(drop=True),
        'source_run': str(FIRE_2D_RUN),
    }

fit_2d_average = _rescale_fit_to_p50(
    pooled_fit, F_2D_MIDPOINT_AVERAGE,
)


def _state_2d_average(fire, threshold):
    return cumulative_top_n_component_state(
        networks_2d[fire], threshold, analysis,
        n_neighbors=SEN_N_NEIGHBORS,
        edge_floor_fraction=SEN_EDGE_FLOOR_ALPHA,
    )


def _fate_2d_average(method_fit):
    return cumulative_top_n_shared_fate_curve(
        networks_2d, analysis, method_fit,
        n_neighbors=SEN_N_NEIGHBORS,
        probabilities=np.arange(.05, .901, .025),
        n_shuffle=N_FATE_SHUFFLES, seed=FATE_SEED,
        edge_floor_fraction=SEN_EDGE_FLOOR_ALPHA,
    )


tag_2d_average = 'ctop2_2d_midpoint_average'
ALTERNATIVE_METHODS[tag_2d_average] = {
    'label': '2D cumulative Top-2',
    'rule': (
        f'α={SEN_EDGE_FLOOR_ALPHA:g}; P50 anchor is mean of direct 2D '
        f'({F_2D_DIRECT_MIDPOINT:.3f}) and inverted 3D '
        f'({F_2D_INVERTED_3D_MIDPOINT:.3f}) midpoints'
    ),
    'fit': fit_2d_average,
}
alternative_results[tag_2d_average] = _threshold_size_result(
    fit_2d_average, _state_2d_average, _fate_2d_average,
)
alternative_results[tag_2d_average]['shared_fate'].to_csv(
    DERIVED / f'fig4c_shared_fate_source_{tag_2d_average}.csv',
    index=False,
)
alternative_results[tag_2d_average]['size_sensitivity'].to_csv(
    DERIVED / f'fig4a_component_size_threshold_sensitivity_{tag_2d_average}.csv',
    index=False,
)
fig_2d_average_sensitivity = plot_alternative_threshold_sensitivity(
    tag_2d_average,
)
display(fig_2d_average_sensitivity)
plt.close(fig_2d_average_sensitivity)

In [ ]:
# Robustness copy: geometry-only distance SENs. The edge score is an
# inverse-square distance proxy normalized so score >= 1 is exactly the
# requested footprint-gap rule of <= 30 ft at P50. P25 and P75 therefore give
# a symmetric, monotone sensitivity analysis around the fixed 30-ft rule.
import shapely

import src.analysis.sen as sen_analysis

DISTANCE_REFERENCE_FT = 30.0
distance_fit = _rescale_fit_to_p50(pooled_fit, 1.0)
distance_cutoffs_ft = {
    probability: DISTANCE_REFERENCE_FT / np.sqrt(
        coupling_at_probability(distance_fit, probability)
    )
    for probability in SIZE_SENSITIVITY_PROBABILITIES
}
distance_max_ft = DISTANCE_REFERENCE_FT / np.sqrt(
    coupling_at_probability(distance_fit, .05)
)
distance_networks = {}
for fire in FIRES:
    nodes = networks[fire]['nodes'].copy()
    geometries = nodes.geometry.to_numpy()
    u, v = sen_analysis.distance_pairs(
        geometries, distance_max_ft * 0.3048,
    )
    gap_m = shapely.distance(geometries[u], geometries[v])
    gap_ft = np.maximum(gap_m / 0.3048, .01)
    edges = pd.DataFrame({
        'bld_a': nodes.BLD_ID.to_numpy()[u],
        'bld_b': nodes.BLD_ID.to_numpy()[v],
        'u': u.astype(np.int64),
        'v': v.astype(np.int64),
        'F_ij': (DISTANCE_REFERENCE_FT / gap_ft) ** 2,
        'ssd_m': gap_m,
        'has_contact': gap_m <= .01,
    })
    distance_networks[fire] = {
        **networks[fire], 'nodes': nodes, 'edges': edges,
        'max_distance_m': distance_max_ft * 0.3048,
        'source_run': 'footprint geometry',
    }


def _state_distance(fire, threshold):
    return component_state(
        distance_networks[fire], threshold, analysis,
    )


def _fate_distance(method_fit):
    return shared_fate_curve(
        distance_networks, analysis, method_fit,
        probabilities=np.arange(.05, .901, .025),
        n_shuffle=N_FATE_SHUFFLES, seed=FATE_SEED,
    )


tag_distance = 'distance_30ft'
ALTERNATIVE_METHODS[tag_distance] = {
    'label': 'Distance-only SENs',
    'rule': (
        r'$F_d=(30\ \mathrm{ft}/gap)^2$; '
        r'$F_d \geq 1$ is exactly $gap \leq 30$ ft'
    ),
    'fit': distance_fit,
}
alternative_results[tag_distance] = _threshold_size_result(
    distance_fit, _state_distance, _fate_distance,
)
alternative_results[tag_distance]['shared_fate'].to_csv(
    DERIVED / f'fig4c_shared_fate_source_{tag_distance}.csv',
    index=False,
)
distance_size_sensitivity = alternative_results[
    tag_distance
]['size_sensitivity'].copy()
distance_size_sensitivity['distance_cutoff_ft'] = (
    distance_size_sensitivity.probability_equivalent.map(
        distance_cutoffs_ft
    )
)
distance_size_sensitivity.to_csv(
    DERIVED / f'fig4a_component_size_threshold_sensitivity_{tag_distance}.csv',
    index=False,
)
print(
    'Distance sensitivity cutoffs:',
    ', '.join(
        f'P{int(probability * 100)}={cutoff:.1f} ft'
        for probability, cutoff in distance_cutoffs_ft.items()
    ),
)
fig_distance_sensitivity = plot_alternative_threshold_sensitivity(
    tag_distance,
)
display(fig_distance_sensitivity)
plt.close(fig_distance_sensitivity)